# 01 — The modelling frame, and why it is not split on time

**ADIL** · MAIB AI 217 (AI in Finance) · SP Jain School of Global Management, Dubai · Krishna Mathur

This notebook narrates artifacts it does not compute. `scripts/build_features.py` runs the
aggregation over all seven Home Credit tables and writes the frame, the manifest, the filter
report and the split assignment; everything below reads those files. Rerunning this notebook
cannot change a number.

It answers three questions an examiner is entitled to ask before any model is fitted:

1. Why is this not split on time?
2. Where could information from after the decision reach the model, and what stops it?
3. Can every one of the 568 columns say where it came from?

The notebook writes `reports/data_quality.md`.

In [ ]:
import json

import pandas as pd

from adil import features, paths, split

pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 150)

processed = paths.processed_dir()
frame = pd.read_parquet(processed / "adil_frame.parquet")
manifest = pd.read_parquet(processed / "feature_manifest.parquet")
filters = pd.read_parquet(processed / "filter_report.parquet")
assignment = pd.read_parquet(processed / "split_index.parquet")
summary = json.loads((paths.metrics_dir() / "frame.json").read_text())

frame.shape

## 1. There is no time axis to split on

`CLAUDE.md` says never to random-split temporal data, and the project brief called for a
temporal split via `spine.splitting`. Neither applies here, and the reason is a property of
the data rather than a preference.

`application_train.csv` has no application date. Every temporal column in it is measured
*relative* to the application, which is exactly the information a date would carry minus the
origin. Below is every column whose name suggests time, with its range — all negative, all
relative, none absolute.

In [ ]:
time_like = [c for c in frame.columns if c.startswith(("DAYS_", "MONTHS_", "HOUR_", "WEEKDAY_"))]
application_time_like = [
    c
    for c in time_like
    if manifest.loc[manifest["feature"] == c, "source_table"].eq("application_train").any()
]


def _range(column):
    series = frame[column]
    return (series.min(), series.max()) if series.dtype.kind in "if" else (None, None)


ranges = [_range(c) for c in application_time_like]
evidence = pd.DataFrame(
    {
        "column": application_time_like,
        "dtype": [str(frame[c].dtype) for c in application_time_like],
        "min": [low for low, _ in ranges],
        "max": [high for _, high in ranges],
    }
)
datetime_columns = [c for c in frame.columns if frame[c].dtype.kind == "M"]
print(f"columns with a datetime dtype anywhere in the frame: {len(datetime_columns)}")
evidence

`HOUR_APPR_PROCESS_START` and `WEEKDAY_APPR_PROCESS_START` are the closest the file comes to
a timestamp, and they are an hour-of-day and a day-of-week with no week attached. They cannot
order two applications.

So a rolling-origin split has no origin. Constructing one — from row order, from `SK_ID_CURR`,
from bureau recency — would manufacture an ordering that is not the application ordering, and
every out-of-time claim built on it would be false. **`spine.splitting` is deliberately unused
in ADIL.** The split is stratified, the exception is documented here, and the seed is recorded.

What a temporal split exists to prevent is handled separately, and more strictly, in section 2.

## 2. The as-of-application discipline

Every satellite aggregate is restricted to records knowable when the application was decided.
Each restriction is an explicit `WHERE` clause in `adil.features.SATELLITES`, and the count of
rows it removes is recorded below — **including where it removes none**. A filter that turns
out to be a no-op is evidence; a filter that was never measured is a hole in the audit trail.

In [ ]:
shown = ["table", "as_of_filter", "rows_before", "rows_kept", "rows_removed", "share_removed"]
report = filters[shown].copy()
report["share_removed"] = (100 * report["share_removed"]).round(4)
report.rename(columns={"share_removed": "removed_%"}, inplace=True)
report

Three of the six filters remove nothing, and three remove something. Both outcomes are results.

The one that matters most is `installments_payments`. Its rule requires **both** `DAYS_INSTALMENT`
and `DAYS_ENTRY_PAYMENT` to be historic, and the rows it drops are instalments that were due
before the application but *paid on or after it*. Those rows describe repayment behaviour that
had not happened when the decision was taken. Filtering on the due date alone would have let
them through, and the model would have learned from the future.

`bureau_balance` is counted after its join to an already-filtered `bureau`, so its `rows_before`
is not the raw file's row count. `adil.features.filter_report` records that distinction in its
`counted_over` column rather than leaving the number to be misread.

In [ ]:
for row in filters.itertuples():
    table = next(t for t in features.SATELLITES if t.name == row.table)
    print(row.table)
    print(f"  filter:       {row.as_of_filter}")
    print(f"  counted over: {row.counted_over}")
    print(f"  why:          {table.as_of_rationale}\n")

## 3. Provenance: every column can name its origin

`tests/test_features.py` enforces that a feature which cannot produce a manifest row cannot
enter the frame. The manifest is the artifact an examiner reads to ask "where did this number
come from", and it is what the model card's data section is built from.

In [ ]:
by_table = (
    manifest.groupby("source_table")
    .agg(features=("feature", "size"), aggregations=("aggregation", "nunique"))
    .sort_values("features", ascending=False)
)
by_table.loc["TOTAL"] = [by_table["features"].sum(), manifest["aggregation"].nunique()]
print(f"columns in frame: {frame.shape[1]}   manifest rows: {len(manifest)}")
print(f"columns without a manifest row: {summary['columns_without_a_manifest_row']}")
by_table

In [ ]:
manifest.sample(8, random_state=split.SEED)[
    ["feature", "source_table", "source_column", "aggregation", "as_of_filter"]
]

## 4. Data quality

Three things that would quietly damage later notebooks if left undiscovered: columns with no
variance (which break weight-of-evidence binning), missingness heavy enough to make a bin
meaningless, and the state of the two attributes the fairness audit depends on.

In [ ]:
missing = frame.isna().mean().sort_values(ascending=False)
constant = frame.columns[frame.nunique(dropna=True) <= 1].tolist()
print(f"all-null columns:  {int((missing == 1).sum())}")
print(f"constant columns:  {len(constant)}  ->  {constant}")
print(f"columns >90% missing: {int((missing > 0.90).sum())}")
print(f"columns >50% missing: {int((missing > 0.50).sum())}")
missing.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).round(4)

### The protected attributes

The project brief assumed real protected attributes were unavailable and that proxies would
stand in. That is not the case here, and the correction matters in both directions.

`CODE_GENDER` is **sex** and `DAYS_BIRTH` is **age**. Both are genuinely protected under
essentially every consumer credit regime — they are not proxies, and the fairness audit in
notebook 05 is a real audit that will be reported as one.

What is absent is **nationality and ethnicity**, which in a UAE context is the axis that matters
most. Home Credit offers no honest proxy for it. That dimension is not analysed here and is
declared unaddressable rather than approximated.

Neither attribute enters the feature set. Using sex or age as a model input would be disparate
treatment; both are held aside purely for measurement. This does **not** make the model fair —
proxies for both survive in the remaining 566 columns — and measuring the disparity that
persists after their removal is itself one of the findings notebook 05 reports.

In [ ]:
gender = frame["CODE_GENDER"].value_counts(dropna=False)
age_years = (-frame["DAYS_BIRTH"] / 365.25).round(1)
print("CODE_GENDER:")
print(gender.to_string())
print(f"\n  'XNA' rows: {int(gender.get('XNA', 0))} — too few to support a group rate.")
print("  They stay in the modelling data and are excluded from the sex disparity")
print("  computation in notebook 05, which will say so.")
print(
    f"\nage (years, from DAYS_BIRTH): min {age_years.min()}"
    f"  median {age_years.median()}  max {age_years.max()}"
)

## 5. Split integrity

Stratified 60/20/20 on the target. Three ways rather than two because the challenger needs a
calibration set the gradient booster has never seen — calibrating on training predictions
produces a reliability curve that flatters the model, and a cost-based threshold built on it is
wrong in a direction nobody notices.

In [ ]:
rates = assignment.groupby("split")["TARGET"].agg(n="size", target_rate="mean")
rates["share"] = (rates["n"] / rates["n"].sum()).round(4)
rates["target_rate"] = rates["target_rate"].round(6)

groups = {name: set(part["SK_ID_CURR"]) for name, part in assignment.groupby("split")}
overlaps = {
    f"{a}&{b}": len(groups[a] & groups[b])
    for a, b in [("train", "calibration"), ("train", "test"), ("calibration", "test")]
}
print(f"seed: {summary['seed']}")
print(f"identifier overlap between splits: {overlaps}")
print(f"target rate spread across splits:  {summary['target_rate_spread']:.2e}  (check: < 1e-3)")
rates

## 6. Write the report

In [ ]:
dropped_instalments = int(filters.set_index("table").loc["installments_payments", "rows_removed"])
constant_list = ", ".join(f"`{c}`" for c in constant) if constant else "none"
skipped = summary["categoricals_skipped_for_cardinality"]
skipped_list = ", ".join(f"`{c}`" for c in skipped) or "none"
sex_counts = ", ".join(f"{k} = {v:,}" for k, v in gender.items())
xna = int(gender.get("XNA", 0))

lines = [
    "# ADIL — data quality and split",
    "",
    "Generated by `notebooks/01_frame.ipynb` from artifacts written by",
    "`scripts/build_features.py`. Every number below is read from a file, not typed.",
    "",
    "MAIB AI 217 · SP Jain School of Global Management, Dubai · Krishna Mathur",
    "",
    "## Source",
    "",
    "Home Credit Default Risk (Kaggle), all seven tables. Public competition data, not UAE",
    "consumer data — the distribution differs materially from any Emirati lending book and no",
    "result here transfers to one without re-estimation.",
    "",
    f"Modelling frame: **{summary['rows']:,} applications × {summary['columns']:,} columns**.",
    f"Overall target rate: **{summary['target_rate_overall']:.5f}**.",
    "",
    "## Why the split is not temporal",
    "",
    "`application_train.csv` carries no application date. Every temporal column in it",
    "(`DAYS_BIRTH`, `DAYS_EMPLOYED`, `DAYS_REGISTRATION`, `DAYS_ID_PUBLISH`,",
    "`DAYS_LAST_PHONE_CHANGE`) is measured relative to the application, and no column in the",
    "frame carries a datetime dtype. `HOUR_APPR_PROCESS_START` and",
    "`WEEKDAY_APPR_PROCESS_START` are an hour-of-day and a day-of-week with no week attached;",
    "they cannot order two applications.",
    "",
    "A rolling-origin split therefore has no origin. Constructing one would manufacture an",
    "ordering that is not the application ordering, and every out-of-time claim built on it",
    "would be false. **`spine.splitting` is deliberately unused in this project.** The split is",
    f"stratified 60/20/20 on the target under recorded seed **{summary['seed']}**.",
    "",
    "## As-of-application filters",
    "",
    "What a temporal split exists to prevent is handled instead by restricting every satellite",
    "aggregate to records knowable when the application was decided. Row counts are reported",
    "whether or not the filter removed anything.",
    "",
    "| Table | Filter | Rows before | Removed | % |",
    "|---|---|---:|---:|---:|",
]
for row in filters.itertuples():
    lines.append(
        f"| `{row.table}` | `{row.as_of_filter}` | {row.rows_before:,} | "
        f"{row.rows_removed:,} | {100 * row.share_removed:.4f} |"
    )
lines += [
    "",
    "Three filters remove nothing and three remove something; both are results. The",
    "`installments_payments` rule requires both `DAYS_INSTALMENT` and `DAYS_ENTRY_PAYMENT` to",
    f"be historic, and the {dropped_instalments:,} rows it drops are instalments due before the",
    "application but paid on or after it — repayment behaviour that had not happened when the",
    "decision was taken. Filtering on the due date alone would have leaked it.",
    "",
    "`bureau_balance` rows are counted after the join to an already-filtered `bureau`, so its",
    "`rows before` is not the raw file's row count.",
    "",
    "## Provenance",
    "",
    f"All {summary['columns']:,} columns have a manifest row naming source table, source column,",
    "aggregation and the filter in force. `tests/test_features.py` fails the suite if any column",
    "cannot. Features by source table:",
    "",
    "| Source table | Features |",
    "|---|---:|",
]
for table, count in manifest.groupby("source_table").size().sort_values(ascending=False).items():
    lines.append(f"| `{table}` | {count} |")
lines += [
    "",
    "## Data quality",
    "",
    f"- All-null columns: **{int((missing == 1).sum())}**",
    f"- Constant columns: **{len(constant)}** — {constant_list}",
    f"- Columns above 90% missing: **{int((missing > 0.90).sum())}**",
    f"- Columns above 50% missing: **{int((missing > 0.50).sum())}**",
    "",
    f"Categorical columns skipped for cardinality above "
    f"{features.CATEGORY_LEVEL_CAP}: {skipped_list}.",
    "",
    "## Protected attributes",
    "",
    "The project brief assumed protected attributes were unavailable and proxies would stand in.",
    "That is wrong in both directions and the correction is load-bearing.",
    "",
    "`CODE_GENDER` is **sex** and `DAYS_BIRTH` is **age**. Both are genuinely protected under",
    "essentially every consumer credit regime. They are not proxies, and the audit in notebook",
    "05 is a real fairness audit reported as one.",
    "",
    "**Nationality and ethnicity are absent**, and in a UAE context that is the axis that matters",
    "most. Home Credit offers no honest proxy for it. That dimension is declared unaddressable",
    "in this dataset rather than approximated.",
    "",
    "Neither attribute enters the feature set — using sex or age as a model input would be",
    "disparate treatment. Both are retained for measurement only. This does not make the model",
    "fair: proxies survive in the remaining columns, and the disparity that persists after their",
    "removal is one of the findings notebook 05 reports.",
    "",
    f"Sex: {sex_counts}. The {xna} `XNA` rows are too few to support a group",
    "rate; they remain in the",
    "modelling data and are excluded from the sex disparity computation, which says so.",
    "",
    f"Age: {age_years.min():.0f} to {age_years.max():.0f} years, median {age_years.median():.0f}.",
    "",
    "## Split",
    "",
    "| Split | n | Share | Target rate |",
    "|---|---:|---:|---:|",
]
for name, row in rates.iterrows():
    lines.append(f"| {name} | {int(row['n']):,} | {row['share']:.4f} | {row['target_rate']:.6f} |")
lines += [
    "",
    f"Identifier overlap between splits: {overlaps} — disjoint.",
    f"Target-rate spread across splits: {summary['target_rate_spread']:.2e}.",
    "",
    "## Limitations",
    "",
    "- Public competition data, not UAE consumer data. The distribution differs materially.",
    "- No time axis, so no out-of-time validation is possible and none is claimed. Metric",
    "  stability over time is untested here and would be a monitoring requirement in production.",
    "- Nationality and ethnicity are not present and are not proxied.",
    "- Amounts are in an anonymised, unscaled currency. All costs are reported in dataset",
    "  currency units; any AED figure in this project is a labelled scenario, not a finding.",
    "",
]
path = paths.reports_dir() / "data_quality.md"
path.write_text("\n".join(lines) + "\n")
print(f"wrote {path}  ({len(lines)} lines)")